# Translation ambiguity in nanobeam ptychography

Each reconstruction is registered against the ground-truth potential; the recorded shift is one
row of a `shifts_conv*_{kind}.csv`. When the diffracted disks do **not** overlap the reconstruction
is only determined up to a translation within the projected unit cell, so the shift is a random
vector uniformly distributed over that cell. Once the disks overlap the solution is unique and the
shift vanishes.

The overlap threshold depends on the zone axis, $\alpha_c = \lambda / 2d$ for the first allowed
ZOLZ reflection:

| particle | zone | reflection | $d$ | $\alpha_c$ |
|---|---|---|---|---|
| single NP, two-NP upper | [111] | {220} | 1.442 Å | **6.8 mrad** |
| two-NP lower | [100] | {200} | 2.039 Å | **4.8 mrad** |

so the two particles in the *same* reconstruction are predicted to become unique at different
convergence angles. All analysis lives in [`ambiguity_analysis.py`](ambiguity_analysis.py).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import ambiguity_analysis as sa

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9, "figure.dpi": 140,
    "savefig.bbox": "tight", "text.color": sa.INK, "axes.labelcolor": sa.INK_MUTED,
})

# (sample, on amorphous carbon, row label) -- one row of the primary figure each
ROWS = [
    ("single_np", False, "single NP\n[111], vacuum"),
    ("single_np", True,  "single NP\n[111], on aC"),
    ("two_upper", False, "two NPs, upper\n[111]"),
    ("two_lower", False, "two NPs, lower\n[100]"),
]
ANGLES = sa.CONVERGENCE_ANGLES

data, meta = {}, {}
for key, on_aC, _ in ROWS:
    runs, offset, scale = sa.build(sa.SAMPLES[key], on_aC)
    data[(key, on_aC)] = runs
    meta[(key, on_aC)] = (offset, scale)

## Calibration

Two numbers per dataset used to be hand-tuned in the old notebooks; both are measured here.

**Registration offset.** At 12 mrad the solution is unique, so whatever shift survives is
instrumental. `calibrate_offset` takes it from that run and refuses to proceed if the run is not
in fact deterministic. (These reproduce the old hard-coded `correction_x/correction_y`.)

**Cell scale.** The old `4.2 px` per unit cell drifts between cells (4.0 / 4.18 / 4.2) and cannot
be right for both fields of view. `measure_cell_scale` reads it off the reconstruction images
instead: the strongest projected fringes turn out to have exactly the spacing of the first allowed
ZOLZ reflection -- the same reflection that sets $\alpha_c$ -- and the measurement is stable to
0.3% across every saved image of a given particle.

In [ ]:
rows = []
for key, on_aC, _ in ROWS:
    s = sa.SAMPLES[key]
    offset, scale = meta[(key, on_aC)]
    _, spread = sa.calibrate_offset(s, on_aC)
    rows.append({
        "sample": s.label, "on_aC": on_aC,
        "offset_x (px)": offset[0], "offset_y (px)": offset[1],
        "12 mrad p95 spread (px)": spread,
        "cell (px)": scale, "cell from fit (px)": sa.fit_cell_scale(s, on_aC),
        "legacy (px)": 4.2,
        "implied pixel size (A)": s.d_min / scale,
        "alpha_c (mrad)": s.alpha_critical,
    })
pd.DataFrame(rows).round(3)

### Two independent checks on the cell scale

The fit above assumes the 1.5 and 3 mrad runs are *fully* non-unique. Two things test that
without making the assumption:

1. the lattice period measured straight off a reconstruction image (single-particle images only —
   the two-particle field of view mixes both orientations);
2. the two particles in the two-NP simulation share a pixel size, so their fitted scales must be
   in the ratio $d_{\rm nn}[111] / d_{\rm nn}[100] = \sqrt2$. With only ~30 seeds at 3 mrad this
   is the noisier of the two checks.

In [ ]:
for key, on_aC, _ in ROWS:
    s = sa.SAMPLES[key]
    scale = meta[(key, on_aC)][1]
    print(f"{s.label:22s} aC={on_aC!s:5s} cell {scale:.2f} px | fit {sa.fit_cell_scale(s, on_aC):.2f} px"
          f" | {s.unit_name.strip('$')} = {s.d_min:.3f} A -> {s.d_min / scale:.4f} A/px")

## Figure 1 — the ambiguity in the reconstructions themselves

Only rendered PNGs of a few representative seeds were kept, not the object arrays, so each figure
is cropped to its image area and resampled back onto the 256 px object grid. That puts runs saved
with different colorbar widths on one common grid; two 12 mrad seeds recovered this way agree to
0.02 px, which bounds the error it introduces.

The 12 mrad reconstruction is the unique solution, so its column positions are the truth, and the
same positions are drawn on every magnified panel. The displacement quoted under each panel is
measured from the image itself, by comparing the phase of the same two fringe families against the
reference — it agrees with the independently registered CSV values to about 0.1 px, and unlike
them it is available for the two seeds whose images were saved but whose rows are not in the CSVs.

These are the amorphous-carbon runs, which reconstruct more cleanly than the vacuum ones.

In [ ]:
import glob
from matplotlib.patches import Rectangle

ON_AC = True
upper, lower = sa.SAMPLES["two_upper"], sa.SAMPLES["two_lower"]

def pngs(sample, angle):
    return sorted(glob.glob(f"{sa.run_dir(sample, angle, ON_AC)}/reconstruction_*.png"),
                  key=sa.seed_of)

ref_png = [p for p in pngs(upper, 12) if sa.seed_of(p) == 0][0]
columns = [(12, ref_png)] + [(a, p) for a in (6, 3) for p in pngs(upper, a)]

ref = sa.load_object_image(ref_png)
bands = sa.particle_bands(ref)
scales = {s.key: sa.measure_cell_scale(s, ON_AC) for s in (upper, lower)}
lattices = {s.key: sa.reference_lattice(ref, band, scales[s.key])
            for s, band in zip((upper, lower), bands)}

H, W = ref.shape
HALF_Y, HALF_X = int(H * 0.75 / 2), int(W * 0.5 / 2)   # 75% tall, 50% wide
ZOOM = 17

fig = plt.figure(figsize=(1.55 * len(columns), 6.6))
gs = fig.add_gridspec(3, len(columns), height_ratios=[HALF_Y / HALF_X, 1, 1],
                      hspace=0.14, wspace=0.06, top=0.855)

for col, (angle, png) in enumerate(columns):
    img = sa.load_object_image(png)
    seed = sa.seed_of(png)
    is_ref = png == ref_png

    ax = fig.add_subplot(gs[0, col])
    sa.plot_recon_crop(ax, img, (H // 2, W // 2), (HALF_Y, HALF_X))
    ax.set_title(f"{angle:g} mrad, seed {seed}" + ("\n(unique reference)" if is_ref else ""),
                 fontsize=8, color=sa.INK, pad=4)
    for band in bands:
        cy = (band[0] + band[1]) // 2
        ax.add_patch(Rectangle((W // 2 - ZOOM - 0.5, cy - ZOOM - 0.5), 2 * ZOOM, 2 * ZOOM,
                               fill=False, ec="#eb6834", lw=0.8, zorder=4))

    for row, (s, band) in enumerate(zip((upper, lower), bands), start=1):
        ax = fig.add_subplot(gs[row, col])
        center = ((band[0] + band[1]) // 2, W // 2)
        sa.plot_recon_crop(ax, img, center, ZOOM, lattices[s.key], marker_size=34)
        shift = np.linalg.norm(sa.lattice_shift(img, ref, band, s.basis(scales[s.key])))
        unique = shift < sa.UNIQUE_BELOW_PX
        ax.text(0.5, -0.04, "reference" if is_ref else f"{shift:.2f} px",
                transform=ax.transAxes, ha="center", va="top", fontsize=7.5,
                color=sa.INK_MUTED if is_ref else (sa.INK if unique else "#eb6834"),
                fontweight="bold" if (unique and not is_ref) else "normal")
        if col == 0:
            ax.set_ylabel(s.label.split(", ")[1], fontsize=8, color=sa.INK)

fig.suptitle("Reconstructions on amorphous carbon, seed by seed", y=0.985,
             fontsize=11, color=sa.INK)
fig.text(0.5, 0.958,
         "circles mark the column positions of the unique 12 mrad solution, identical in every magnified panel;  "
         "boxes mark the magnified regions\n"
         "below 6.8 mrad the [111] lattice sits off them; below 4.8 mrad the [100] one does too",
         ha="center", va="top", fontsize=7.6, color=sa.INK_MUTED)

for ext in ("pdf", "png"):
    fig.savefig(f"fig1_seed_to_seed_images.{ext}")

## Figure 2 — where the reconstruction lands inside the unit cell

The primary result. Each point is one random seed, folded into the Wigner–Seitz cell of that
particle's projected lattice. Left to right the cell fills uniformly and then collapses onto the
true column position; the [100] row collapses one column earlier than the [111] rows, exactly
where its own disk-overlap threshold sits. Runs with fewer than five seeds are left blank -- one
dot is not a distribution.

In [ ]:
fig, axes = plt.subplots(len(ROWS), len(ANGLES), figsize=(7.2, 7.6))

for i, (key, on_aC, label) in enumerate(ROWS):
    s = sa.SAMPLES[key]
    runs = data[(key, on_aC)]
    for j, angle in enumerate(ANGLES):
        ax = axes[i, j]
        df = runs.get(angle, pd.DataFrame(columns=["dx", "dy", "r"]))
        sa.plot_cell_scatter(ax, df, s, sa.ANGLE_COLORS[angle],
                             disks_overlap=angle >= s.alpha_critical)
        if i == 0:
            ax.set_title(f"{angle:g} mrad", color=sa.INK, pad=6)
        if j == 0:
            ax.text(-0.16, 0.5, label, transform=ax.transAxes, rotation=90,
                    ha="center", va="center", fontsize=8.5, color=sa.INK)

fig.subplots_adjust(top=0.86, hspace=0.12, wspace=0.06)
fig.suptitle("Reconstruction shift within the projected unit cell", y=0.975,
             fontsize=11, color=sa.INK)
fig.text(0.5, 0.928, "one point per random seed;  outline = Wigner-Seitz cell,  "
         "+ = true atom column;  % unique = seeds within 0.1 px of the true solution\n"
         "shaded panels = convergence angle above that particle's disk-overlap "
         "threshold $\\alpha_c = \\lambda / 2d$",
         ha="center", va="top", fontsize=7.8, color=sa.INK_MUTED)

for ext in ("pdf", "png"):
    fig.savefig(f"fig2_shift_cell_grid.{ext}")

## Figure 3 — magnitude distribution and the uniqueness transition

(a) ECDFs rather than histograms: no binning, and comparable across runs whose N differs by 20×.
The 1.5 and 3 mrad curves sit on the uniform-over-cell null, 6 mrad departs from it, 12 mrad is a
step at zero.

(b) The transition lands at each particle's own $\alpha_c$.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(9.4, 3.0))

sa.plot_ecdf(ax1, data[("single_np", False)], sa.SAMPLES["single_np"])
ax1.set_xlabel("|shift|  (cells)")
ax1.set_ylabel("fraction of seeds")
ax1.set_title("a   single NP [111], vacuum", color=sa.INK, loc="left", fontsize=9)
ax1.legend(frameon=False, fontsize=7.5, loc="upper left", labelcolor=sa.INK_MUTED,
           borderaxespad=0.6, handlelength=1.6, labelspacing=0.35)

STYLE = dict(zip([r[0] + str(r[1]) for r in ROWS], ["o", "s", "^", "D"]))
for key, on_aC, label in ROWS:
    s_ = sa.SAMPLES[key]
    runs = data[(key, on_aC)]
    null_median = np.median(sa.uniform_cell_radii(s_.basis(1.0)))
    xs = [a for a in ANGLES if a in runs and len(runs[a]) >= 5]
    kw = dict(marker=STYLE[key + str(on_aC)], ms=5, lw=1.6,
              color="#1c5cab" if s_.zone == "111" else "#eb6834",
              ls="--" if on_aC else "-", label=label.replace("\n", " "))
    ax2.plot(xs, [np.median(runs[a]["r"]) / null_median for a in xs], **kw)
    ax3.plot(xs, [(runs[a]["r_px"] < sa.UNIQUE_BELOW_PX).mean() for a in xs], **kw)

for ax, title, ylabel in [
    (ax2, "b   how ambiguous", "median |shift| /\nnon-unique expectation"),
    (ax3, "c   how often exact", "fraction of seeds within\nthe noise floor"),
]:
    for zone, color, ha in [("100", "#eb6834", "right"), ("111", "#1c5cab", "left")]:
        a_c = next(v.alpha_critical for v in sa.SAMPLES.values() if v.zone == zone)
        ax.axvline(a_c, color=color, lw=1.0, ls=":", zorder=0)
        ax.text(a_c, 0.55, f"  $\\alpha_c$[{zone}] = {a_c:.1f} mrad  ", color=color,
                fontsize=7, ha=ha, va="center", rotation=90,
                bbox=dict(fc="white", ec="none", pad=0.5))
    ax.set_xscale("log")
    ax.set_xticks(ANGLES)
    ax.set_xticklabels([f"{a:g}" for a in ANGLES])
    ax.minorticks_off()
    ax.set_xlabel("convergence semi-angle (mrad)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.06, 1.15)
    ax.set_title(title, color=sa.INK, loc="left", fontsize=9)
    sa.style_axes(ax)

ax2.axhline(1.0, color=sa.NULL_GRAY, ls="--", lw=1.2, zorder=0)
handles, labels = ax2.get_legend_handles_labels()
fig.subplots_adjust(wspace=0.45, bottom=0.30)
fig.legend(handles, labels, frameon=False, fontsize=7.5, ncol=4,
           loc="lower center", bbox_to_anchor=(0.5, 0.0),
           labelcolor=sa.INK_MUTED, handlelength=1.6, columnspacing=1.8)

for ext in ("pdf", "png"):
    fig.savefig(f"fig3_magnitude_and_transition.{ext}")

## Figure 4 — direction carries no information (supplementary)

The ambiguity has no preferred direction: the Rayleigh test never rejects uniformity. At 12 mrad
the shift is zero, so its direction is undefined rather than uniform — which is why this is a
supporting panel and not the main figure.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(7.6, 2.3), subplot_kw={"projection": "polar"})

runs = data[("single_np", False)]
for ax, angle in zip(axes, ANGLES):
    df = runs[angle]
    sa.plot_rose(ax, df, sa.ANGLE_COLORS[angle])
    n, z, p = sa.rayleigh_test(df["theta_deg"], df["r"])
    note = f"Rayleigh p = {p:.2f}" if np.isfinite(p) else "no direction (unique)"
    ax.set_title(f"{angle:g} mrad\n{note}", fontsize=8, color=sa.INK, pad=8)

fig.subplots_adjust(top=0.62, wspace=0.55)
fig.text(0.5, 0.99, "Shift direction is uniformly distributed at every convergence angle",
         ha="center", va="top", fontsize=9.5, color=sa.INK)
fig.text(0.5, 0.90, "single NP [111], vacuum;  dashed circle = uniform expectation",
         ha="center", va="top", fontsize=7.5, color=sa.INK_MUTED)

for ext in ("pdf", "png"):
    fig.savefig(f"fig4_direction_rose.{ext}")

## Summary table

`unique_frac` is the fraction of seeds within 0.05 cells of the true solution; `ks_vs_null` is the
distance from the fully-non-unique null (small = indistinguishable from complete ambiguity,
1 = perfectly unique); `wrapped_frac` is the fraction of rows that registration had locked onto a
neighbouring lattice peak, folded back into the cell here.

In [ ]:
summary = pd.DataFrame([
    sa.summarize(df, sa.SAMPLES[key])
    for key, on_aC, _ in ROWS
    for df in data[(key, on_aC)].values()
])
summary.to_csv("ambiguity_summary.csv", index=False)
summary.round(3)